In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/data.csv')
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])
print(f"Loaded: {df.shape}")

Loaded: (95662, 16)


In [2]:
# Create aggregate features per customer
agg_features = df.groupby('CustomerId').agg(
    total_transactions=('TransactionId', 'count'),
    total_amount=('Amount', 'sum'),
    avg_amount=('Amount', 'mean'),
    std_amount=('Amount', 'std'),
    max_amount=('Amount', 'max'),
    min_amount=('Amount', 'min'),
    total_value=('Value', 'sum'),
    unique_products=('ProductId', 'nunique'),
    unique_channels=('ChannelId', 'nunique'),
    fraud_count=('FraudResult', 'sum')
).reset_index()

agg_features['std_amount'] = agg_features['std_amount'].fillna(0)
print(f"Aggregate features: {agg_features.shape}")
print(agg_features.head())

Aggregate features: (3742, 11)
        CustomerId  total_transactions  total_amount    avg_amount  \
0     CustomerId_1                   1      -10000.0 -10000.000000   
1    CustomerId_10                   1      -10000.0 -10000.000000   
2  CustomerId_1001                   5       20000.0   4000.000000   
3  CustomerId_1002                  11        4225.0    384.090909   
4  CustomerId_1003                   6       20000.0   3333.333333   

    std_amount  max_amount  min_amount  total_value  unique_products  \
0     0.000000    -10000.0    -10000.0        10000                1   
1     0.000000    -10000.0    -10000.0        10000                1   
2  6558.963333     10000.0     -5000.0        30400                3   
3   560.498966      1500.0       -75.0         4775                3   
4  6030.478146     10000.0     -5000.0        32000                4   

   unique_channels  fraud_count  
0                1            0  
1                1            0  
2            

In [3]:
reference_date = df['TransactionStartTime'].max()

rfm = df.groupby('CustomerId').agg(
    Recency=('TransactionStartTime', lambda x: (reference_date - x.max()).days),
    Frequency=('TransactionId', 'count'),
    Monetary=('Amount', 'sum')
).reset_index()

rfm['Recency_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4,3,2,1]).astype(int)
rfm['Frequency_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['Monetary_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['RFM_Score'] = rfm['Recency_Score'] + rfm['Frequency_Score'] + rfm['Monetary_Score']
rfm['Risk_Label'] = (rfm['RFM_Score'] <= rfm['RFM_Score'].quantile(0.33)).astype(int)

print(f"High Risk: {rfm['Risk_Label'].sum()} ({rfm['Risk_Label'].mean()*100:.1f}%)")

High Risk: 1508 (40.3%)


In [4]:
# Merge all features
final_df = agg_features.merge(rfm[['CustomerId','Recency','Frequency','Monetary','RFM_Score','Risk_Label']], on='CustomerId')

# Save
final_df.to_csv('../data/processed/features.csv', index=False)
print(f"Saved! Shape: {final_df.shape}")
print(final_df.head())

Saved! Shape: (3742, 16)
        CustomerId  total_transactions  total_amount    avg_amount  \
0     CustomerId_1                   1      -10000.0 -10000.000000   
1    CustomerId_10                   1      -10000.0 -10000.000000   
2  CustomerId_1001                   5       20000.0   4000.000000   
3  CustomerId_1002                  11        4225.0    384.090909   
4  CustomerId_1003                   6       20000.0   3333.333333   

    std_amount  max_amount  min_amount  total_value  unique_products  \
0     0.000000    -10000.0    -10000.0        10000                1   
1     0.000000    -10000.0    -10000.0        10000                1   
2  6558.963333     10000.0     -5000.0        30400                3   
3   560.498966      1500.0       -75.0         4775                3   
4  6030.478146     10000.0     -5000.0        32000                4   

   unique_channels  fraud_count  Recency  Frequency  Monetary  RFM_Score  \
0                1            0       83     

In [5]:
# Weight of Evidence for categorical features
# We use the transaction level data for WoE
def calculate_woe_iv(df, feature, target):
    df_temp = df[[feature, target]].copy()
    df_temp['total'] = 1
    stats = df_temp.groupby(feature).agg(
        events=(target, 'sum'),
        total=('total', 'count')
    ).reset_index()
    stats['non_events'] = stats['total'] - stats['events']
    total_events = stats['events'].sum()
    total_non_events = stats['non_events'].sum()
    stats['dist_events'] = stats['events'] / total_events
    stats['dist_non_events'] = stats['non_events'] / total_non_events
    stats['WoE'] = np.log((stats['dist_events'] + 0.0001) / (stats['dist_non_events'] + 0.0001))
    stats['IV'] = (stats['dist_events'] - stats['dist_non_events']) * stats['WoE']
    return stats, stats['IV'].sum()

# Apply WoE to ProductCategory
df_labeled = df.merge(rfm[['CustomerId','Risk_Label']], on='CustomerId')
woe_stats, iv = calculate_woe_iv(df_labeled, 'ProductCategory', 'Risk_Label')
print(f"ProductCategory IV: {iv:.4f}")
print(woe_stats[['ProductCategory','WoE','IV']])

ProductCategory IV: 0.0355
      ProductCategory       WoE        IV
0             airtime -0.122011  0.006634
1        data_bundles  0.199349  0.000738
2  financial_services  0.099028  0.004870
3              movies  1.735251  0.012813
4               other  1.692422  0.000750
5              ticket  1.070169  0.004434
6           transport  0.433020  0.000083
7                  tv -0.650275  0.004284
8        utility_bill -0.223854  0.000914
